In [0]:
spark.sql("SELECT current_catalog()").show()
spark.sql("SHOW CATALOGS").show()
spark.sql("SHOW SCHEMAS").show()


+-----------------+
|current_catalog()|
+-----------------+
|        workspace|
+-----------------+

+---------+
|  catalog|
+---------+
|emissions|
|  samples|
|   system|
|workspace|
+---------+

+------------------+
|      databaseName|
+------------------+
|           default|
|information_schema|
|  movielens_bronze|
|    movielens_gold|
|  movielens_silver|
+------------------+



In [0]:

df_als_input = spark.table("workspace.movielens_gold.als_training_view")

# 2. DROP NULLS - This is the fix for your error!
# We remove any row where userId, movieId, or rating is missing.
df_clean = df_als_input.na.drop(subset=["userId", "movieId", "rating"])

# 3. Re-do the split with the clean data
(training_df, test_df) = df_clean.randomSplit([0.8, 0.2], seed=42)

print(f"Cleaned Rows: {df_clean.count()}")
print(f"New Training Count: {training_df.count()}")
print(f"Test Rows (20%): {test_df.count()}")

Cleaned Rows: 100836
New Training Count: 80481
Test Rows (20%): 20355


In [0]:
from pyspark.ml.recommendation import ALS

# 1. Initialize the ALS model
als = ALS(
    userCol="userId", 
    itemCol="movieId", 
    ratingCol="rating", 
    nonnegative=True,    # We don't want the model to guess "negative" ratings (ratings are 1-5)
    implicitPrefs=False, # We are using real ratings (1-5), not just "clicks"
    coldStartStrategy="drop" # If a user has NO data, don't crash, just ignore them
)

# 2. Train the model using our 80% training data
# This is where the computer does the "Alternating" math
model = als.fit(training_df)
print("🧠 Success! The model trained without hitting any Nulls.")

🧠 Success! The model trained without hitting any Nulls.


In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

# 1. Generate predictions on our Test data
predictions = model.transform(test_df)

# 2. Setup the Evaluator (The "Teacher" that grades the test)
evaluator = RegressionEvaluator(
    metricName="rmse", 
    labelCol="rating", 
    predictionCol="prediction"
)

# 3. Calculate the RMSE
rmse = evaluator.evaluate(predictions)

print(f"📉 Root-Mean-Square Error (RMSE) = {rmse}")

# 4. Look at some actual vs predicted values
display(predictions.select("userId", "movieId", "rating", "prediction").limit(10))

📉 Root-Mean-Square Error (RMSE) = 0.8761925742024357


userId,movieId,rating,prediction
94,2,4.0,2.8723097
94,10,3.0,3.2571301
94,19,2.0,2.3149395
94,21,3.0,2.5388348
94,160,3.0,2.4265096
94,161,4.0,3.4332085
94,225,3.0,3.196049
94,292,3.0,3.2871444
94,296,4.0,3.4405255
94,329,4.0,3.000728


fixing

This creates all possible combinations.

In [0]:
users = df_clean.select("userId").distinct()
movies = df_clean.select("movieId").distinct()

user_movie_pairs = users.crossJoin(movies)


Predict Ratings


In [0]:
predictions = model.transform(user_movie_pairs)
predictions = predictions.na.drop()


Rank Predictions per User

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, desc

window_spec = Window.partitionBy("userId").orderBy(desc("prediction"))

ranked_predictions = predictions.withColumn(
    "rank",
    row_number().over(window_spec)
)


Select Top 5 per User

In [0]:
top5_recommendations = ranked_predictions.filter("rank <= 5")

top5_recommendations.select(
    "userId",
    "movieId",
    "prediction"
).show()


+------+-------+----------+
|userId|movieId|prediction|
+------+-------+----------+
|     1|   5490|  5.737888|
|     1|   5915|  5.737888|
|     1| 132333|  5.737888|
|     1|  33649| 5.7304664|
|     1|   6818| 5.5311627|
|     2| 131724|   4.90274|
|     2| 170355|  4.801151|
|     2|  33649| 4.7637215|
|     2|  86377| 4.6799984|
|     2|  51931|  4.655016|
|     3|  70946| 4.9285693|
|     3|   6835| 4.9123096|
|     3|   5746| 4.9123096|
|     3|   7991| 4.7771726|
|     3|   2851| 4.6985607|
|     4|   1916|  6.044026|
|     4|   6818| 5.4973493|
|     4|  40491|  5.396235|
|     4|  25825|   5.35464|
|     4| 148881| 5.3530235|
+------+-------+----------+
only showing top 20 rows


In [0]:
# Step 1: Load movie metadata from the Silver layer.
# We select only movieId and pure_title and remove duplicates
# so that each movie appears once.

df_movies = (
    spark.table("workspace.movielens_silver.fact_ratings")
         .select("movieId", "pure_title")
         .distinct()
)

df_movies.show(5)


+-------+--------------------+
|movieId|          pure_title|
+-------+--------------------+
|      1|           Toy Story|
|      2|             Jumanji|
|      3|    Grumpier Old Men|
|      4|   Waiting to Exhale|
|      5|Father of the Bri...|
+-------+--------------------+
only showing top 5 rows


In [0]:
# Step 2: Join the Top-5 recommendations with movie titles.
# This enriches the prediction results with readable movie names.

recs_with_titles = top5_recommendations.join(
    df_movies,
    on="movieId",
    how="left"
)

recs_with_titles.show(5)


+-------+------+----------+----+--------------------+
|movieId|userId|prediction|rank|          pure_title|
+-------+------+----------+----+--------------------+
|   5490|     1|  5.737888|   1|         The Big Bus|
|   5915|     1|  5.737888|   2|Victory (a.k.a. E...|
| 132333|     1|  5.737888|   3|                Seve|
|  33649|     1| 5.7304664|   4|         Saving Face|
|   6818|     1| 5.5311627|   5|Come and See (Idi...|
+-------+------+----------+----+--------------------+
only showing top 5 rows


In [0]:
# Step 3: Sort recommendations for each user
# so that the highest predicted ratings appear first.

from pyspark.sql.functions import desc

ordered_recs = recs_with_titles.orderBy(
    "userId",
    desc("prediction")
)

ordered_recs.select(
    "userId",
    "pure_title",
    "prediction"
).show(truncate=False)


+------+------------------------------------------------+----------+
|userId|pure_title                                      |prediction|
+------+------------------------------------------------+----------+
|1     |The Big Bus                                     |5.737888  |
|1     |Victory (a.k.a. Escape to Victory)              |5.737888  |
|1     |Seve                                            |5.737888  |
|1     |Saving Face                                     |5.7304664 |
|1     |Come and See (Idi i smotri)                     |5.5311627 |
|2     |The Jinx: The Life and Deaths of Robert Durst   |4.90274   |
|2     |Mulholland Dr.                                  |4.801151  |
|2     |Saving Face                                     |4.7637215 |
|2     |Louis C.K.: Shameless                           |4.6799984 |
|2     |Reign Over Me                                   |4.655016  |
|3     |Troll 2                                         |4.9285693 |
|3     |Alien Contamination       

In [0]:
# Step 4: Group recommendations by user
# and collect movie titles into a list per user.

from pyspark.sql.functions import collect_list

grouped_recs = ordered_recs.groupBy("userId").agg(
    collect_list("pure_title").alias("recommended_movies")
)

grouped_recs.show(truncate=False)


+------+-----------------------------------------------------------------------------------------------------------------------------------------+
|userId|recommended_movies                                                                                                                       |
+------+-----------------------------------------------------------------------------------------------------------------------------------------+
|1     |[The Big Bus, Victory (a.k.a. Escape to Victory), Seve, Saving Face, Come and See (Idi i smotri)]                                        |
|2     |[The Jinx: The Life and Deaths of Robert Durst, Mulholland Dr., Saving Face, Louis C.K.: Shameless, Reign Over Me]                       |
|3     |[Troll 2, Alien Contamination, Galaxy of Terror (Quest), Death Race 2000, Saturn 3]                                                      |
|4     |[Buffalo '66 (a.k.a. Buffalo 66), Come and See (Idi i smotri), Match Factory Girl, The (Tulitikkutehtaan tyttö

In [0]:
# Step 5: Convert grouped results to Python objects
# and print recommendations in a readable sentence format.

results = grouped_recs.collect()

for row in results:
    print(f"User {row['userId']} – Recommended movies are:")
    for movie in row['recommended_movies']:
        print(f"   - {movie}")
    print()


User 1 – Recommended movies are:
   - The Big Bus
   - Victory (a.k.a. Escape to Victory)
   - Seve
   - Saving Face
   - Come and See (Idi i smotri)

User 2 – Recommended movies are:
   - The Jinx: The Life and Deaths of Robert Durst
   - Mulholland Dr.
   - Saving Face
   - Louis C.K.: Shameless
   - Reign Over Me

User 3 – Recommended movies are:
   - Troll 2
   - Alien Contamination
   - Galaxy of Terror (Quest)
   - Death Race 2000
   - Saturn 3

User 4 – Recommended movies are:
   - Buffalo '66 (a.k.a. Buffalo 66)
   - Come and See (Idi i smotri)
   - Match Factory Girl, The (Tulitikkutehtaan tyttö)
   - Fury
   - World of Tomorrow

User 5 – Recommended movies are:
   - Rivers and Tides
   - Gigantic (A Tale of Two Johns)
   - The Big Bus
   - Victory (a.k.a. Escape to Victory)
   - Seve

User 6 – Recommended movies are:
   - Glory Road
   - Hello, Dolly!
   - Last Detail, The
   - Thief
   - Jezebel

User 7 – Recommended movies are:
   - The Big Bus
   - Victory (a.k.a. Escape t

In [0]:
# Step 9: Save recommendations to a local file on the driver node

from pyspark.sql.functions import concat_ws

# Convert movie lists to comma-separated strings
csv_ready = grouped_recs.withColumn(
    "recommended_movies_str",
    concat_ws(", ", "recommended_movies")
).select("userId", "recommended_movies_str")

# Collect as Pandas to write locally
pdf = csv_ready.toPandas()

# Local path on the driver node
local_path = "/tmp/user_recommendations.csv"

# Save to CSV locally
pdf.to_csv(local_path, index=False)

print(f"Recommendations saved locally at {local_path}")
print("Use the Databricks notebook sidebar or the following command to download the file:")
print(f"%fs cp file:{local_path} dbfs:/FileStore/user_recommendations.csv")


Recommendations saved locally at /tmp/user_recommendations.csv
Use the Databricks notebook sidebar or the following command to download the file:
%fs cp file:/tmp/user_recommendations.csv dbfs:/FileStore/user_recommendations.csv


In [0]:
%sql
-- Replace 'my_schema' with a name you want to use
CREATE SCHEMA IF NOT EXISTS workspace.my_schema;


In [0]:
from pyspark.sql.functions import concat_ws

csv_ready = grouped_recs.withColumn(
    "recommended_movies_str",
    concat_ws(", ", "recommended_movies")
).select("userId", "recommended_movies_str")

# Save to your schema
csv_ready.write.mode("overwrite").saveAsTable("workspace.my_schema.user_recommendations")


In [0]:
from pyspark.sql.functions import col

# Read your Unity Catalog table
df = spark.table("workspace.my_schema.user_recommendations")

# Show the first 20 rows (default)
df.show(truncate=False)

# Optional: show more rows, e.g., first 50
df.show(50, truncate=False)

# Optional: display as a nicely formatted table in Databricks notebook
display(df)


+------+---------------------------------------------------------------------------------------------------------------------------------------+
|userId|recommended_movies_str                                                                                                                 |
+------+---------------------------------------------------------------------------------------------------------------------------------------+
|1     |The Big Bus, Victory (a.k.a. Escape to Victory), Seve, Saving Face, Come and See (Idi i smotri)                                        |
|2     |The Jinx: The Life and Deaths of Robert Durst, Mulholland Dr., Saving Face, Louis C.K.: Shameless, Reign Over Me                       |
|3     |Troll 2, Alien Contamination, Galaxy of Terror (Quest), Death Race 2000, Saturn 3                                                      |
|4     |Buffalo '66 (a.k.a. Buffalo 66), Come and See (Idi i smotri), Match Factory Girl, The (Tulitikkutehtaan tyttö), Fury, Worl

userId,recommended_movies_str
1,"The Big Bus, Victory (a.k.a. Escape to Victory), Seve, Saving Face, Come and See (Idi i smotri)"
2,"The Jinx: The Life and Deaths of Robert Durst, Mulholland Dr., Saving Face, Louis C.K.: Shameless, Reign Over Me"
3,"Troll 2, Alien Contamination, Galaxy of Terror (Quest), Death Race 2000, Saturn 3"
4,"Buffalo '66 (a.k.a. Buffalo 66), Come and See (Idi i smotri), Match Factory Girl, The (Tulitikkutehtaan tyttö), Fury, World of Tomorrow"
5,"Rivers and Tides, Gigantic (A Tale of Two Johns), The Big Bus, Victory (a.k.a. Escape to Victory), Seve"
6,"Glory Road, Hello, Dolly!, Last Detail, The, Thief, Jezebel"
7,"The Big Bus, Victory (a.k.a. Escape to Victory), Seve, Come and See (Idi i smotri), Producers, The"
8,"Rivers and Tides, It Happened One Night, Wild Tales, Best of Youth, The (La meglio gioventù), Play Time (a.k.a. Playtime)"
9,"Once, Grand Day Out with Wallace and Gromit, A, Visitor, The, Frozen River, Come and See (Idi i smotri)"
10,"Melancholia, Snow Dogs, Ghost Town, First Daughter, Ooops! Noah is Gone..."
